In [0]:
# --- CELL 1: SETUP & SECURE CLOUD CONNECTION ---

# ==============================================================================
# ENTERPRISE ARCHITECTURE NOTE:
# In a production enterprise environment, we would securely connect to the AWS S3 
# Data Lake using Databricks Unity Catalog or an AWS IAM Instance Profile. 
# Hardcoding access keys via `spark.conf.set()` is an anti-pattern.
# 
# PORTFOLIO / FREE EDITION LIMITATIONS:
# Databricks Serverless (Free Edition) enforces strict security constraints:
# 1. It blocks manual `fs.s3a` configurations (SQLSTATE: 42K0I) to enforce Unity Catalog.
# 2. The Serverless VPC aggressively drops outbound Boto3 HTTP requests to external S3 buckets.
#
# WORKAROUND APPLIED: 
# To bypass the Serverless egress firewall and demonstrate PySpark transformations 
# at zero cost, the raw JSON files were manually staged into a Databricks Volume (Cell 2).
#
# The commented code below represents the standard PySpark S3 connection string.
# ==============================================================================

"""
# Standard PySpark S3 Configuration (Blocked by Serverless Security)
spark.conf.set("fs.s3a.access.key", "YOUR_ACCESS_KEY")
spark.conf.set("fs.s3a.secret.key", "YOUR_SECRET_KEY")
spark.conf.set("fs.s3a.endpoint", "s3.eu-central-1.amazonaws.com")

bucket_name = "smart-meter-raw-data-luka-frankfurt"
weather_path = f"s3a://{bucket_name}/raw_data/weather/"
pricing_path = f"s3a://{bucket_name}/raw_data/pricing/"

print("Spark cluster connected to AWS S3 Data Lake")
"""

print("Connecting directly to Databricks Volume (Bypassing Serverless Firewall)...")

In [0]:
# --- CELL 2: PYSPARK TRANSFORMATIONS ---
from pyspark.sql.functions import col, explode, to_timestamp, round

# 1. Load the raw JSON from your Databricks Volume
# PASTE YOUR EXACT COPIED PATHS HERE:
raw_weather_df = spark.read.json("/Volumes/workspace/default/lstoric-energy/weather_frankfurt_1775498473.json")
raw_pricing_df = spark.read.json("/Volumes/workspace/default/lstoric-energy/awattar_germany_1775498477.json")

# 2. Flatten the Weather Data
weather_flat = raw_weather_df.select(
    col("current_weather.time").alias("observed_time"),
    col("current_weather.temperature").alias("temp_celsius"),
    col("current_weather.windspeed").alias("windspeed_kmh")
)

# 3. Flatten the Pricing Data
pricing_flat = raw_pricing_df.select(explode(col("data")).alias("price_data")).select(
    to_timestamp(col("price_data.start_timestamp") / 1000).alias("market_hour"),
    round(col("price_data.marketprice"), 2).alias("market_price_mwh")
)

display(pricing_flat) # Let's display the pricing data just to make sure it exploded correctly!

In [0]:
# --- CELL 3: MARKET PRICE VOLATILITY ANALYSIS ---
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Convert Spark DataFrame to Pandas for local Plotting
pdf_pricing = pricing_flat.toPandas()

# 2. Sort the data chronologically
pdf_pricing = pdf_pricing.sort_values(by="market_hour")

print("Analyzing Market Price Volatility...")

# 3. Generate a professional Time-Series Line Chart (Not a Histogram!)
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=pdf_pricing, x='market_hour', y='market_price_mwh', 
             marker='o', color='crimson', linewidth=2, ax=ax)

# 4. Make it look like an Enterprise Dashboard
ax.set_title("Live Energy Market Price Volatility (Awattar Germany)", fontsize=14, fontweight='bold')
ax.set_ylabel("Market Price (€ / MWh)", fontsize=12)
ax.set_xlabel("Market Hour", fontsize=12)

# Format the x-axis to show clean hours
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d - %H:%M'))
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Display the chart
plt.show()